# Exercise 1: Tensor basics 
In this exercise you will learn the basics of tensor creation, manipulation, indexing, broadcasting, vectorization, einsum, and attention masking fundamentals. These basics are important for understanding any complex implementation later on so make sure you understand them well.

**To complete this exercise fill in all TODOs in the functions below.** 

Make sure to check the output of your function and whether or not it fulfills the requirements outlined in the function definition. Do NOT change the function signature or name since we will be running checks on your functions during grading.

### Shape legend used in this notebook
- `B`: batch size
- `T`: sequence length / time
- `D`: feature dimension
- `H`: number of attention heads
- `Dh`: per-head feature dimension

### Debugging tip: what to print
When you get a shape error, print:
- `x.shape`, `x.dtype`, `x.device`
- `x.is_contiguous()` (important for `view`)
For masks also print:
- `mask.shape`, `mask.dtype`, `mask.sum()` and a small slice like `mask[0, :10]`

### Reproducibility tip: seeding in PyTorch
Many operations in deep learning involve randomness (e.g., initializing model weights, shuffling data, dropout, random augmentations).
**Seeding** sets the starting state of PyTorch’s random number generator so that these random choices become **repeatable**.

- If you set the same seed and run the same code again, you should get the same *random* tensors / initial weights.
- If you don’t set a seed, results can vary between runs.

Common usage: `torch.manual_seed(seed)`

Note: even with fixed seeds, some GPU operations can still be non-deterministic due to performance optimizations. For this assignment, seeding is mainly to make debugging easier and to ensure everyone can reproduce the same intermediate results. If you are given a seed, make sure to use it when creating tensors or performing other operations.

## Tensor creation
This warmup exercise teaches you how to create tensors with different shapes and values. A few details about tensor creation that are good to know:
- `torch.tensor([...])` infers dtype from Python values (ints → integer tensor, floats → float tensor).
- `torch.arange(start, end)` is **end-exclusive**.
- `torch.linspace(start, end, steps)` is **end-inclusive**.

In [1]:
from collections.abc import Sequence
import torch

In [2]:
def make_tensor(data, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """ Create a tensor from Python data (list/tuple/nested lists). """
    tensor = torch.tensor(data,dtype=dtype)
    return tensor
    # TODO: implement
    raise NotImplementedError

x = make_tensor([[1, 2], [3, 4]], dtype=torch.float32)

In [3]:
def make_zeros(shape: Sequence[int], dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with zeros."""
    zero = torch.zeros(shape, dtype=dtype)
    return zero
    # TODO: implement
    raise NotImplementedError

z = make_zeros((2, 3), dtype=torch.float64)

In [4]:
def make_ones_like(x: torch.Tensor) -> torch.Tensor:
    ones = torch.ones(x.shape,dtype=x.dtype,device=x.device)
    return ones
    """Create a tensor of ones with the same shape, dtype, and device as x. """
    # TODO: implement
    raise NotImplementedError

base = torch.randn(2, 3, dtype=torch.float32)
ones = make_ones_like(base)

In [5]:
def make_arange(start: int, end: int, step: int = 1, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    arange = torch.arange(start=start,end=end,step=step)
    return arange
    """Create a 1D tensor containing values [start, start+step, ..., < end]."""
    # TODO: implement
    raise NotImplementedError

ar = make_arange(0, 5, 2, dtype=torch.int64)

In [6]:
def make_linspace(start: float, end: float, steps: int, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a 1D tensor with evenly spaced values from start to end (inclusive)."""
    linspace= torch.linspace(start=start,end=end,steps=steps,dtype=dtype,device=device)
    return linspace    
    # TODO: implement
    raise NotImplementedError

ls = make_linspace(0.0, 1.0, steps=5, dtype=torch.float32)

In [7]:
def make_randn(shape: Sequence[int], seed: int | None = None, dtype: torch.dtype | None = None, device: torch.device | str | None = None) -> torch.Tensor:
    """Create a tensor filled with values from a standard normal distribution."""
    generator = torch.Generator(device=device)
    generator.manual_seed(seed)
    randn = torch.randn(size=shape,generator=generator, dtype=dtype)
    return randn    
    # TODO: implement
    raise NotImplementedError

a = make_randn((2, 3), seed=123, dtype=torch.float32)

In [8]:
def cast_dtype_and_move(x: torch.Tensor, device: torch.device, dtype: torch.dtype) -> torch.Tensor:
    """Convert tensor dtype and move to device."""
    x = x.to(device=device, dtype=dtype)
    return x
    # TODO: implement
    raise NotImplementedError

casted = cast_dtype_and_move(torch.tensor([1, 2, 3]), torch.device("cpu"), torch.float32)

## Shape manipulation
Now that we covered the basic tensor creation schemes, we want to focus on shape manipulation. Understanding the difference between these mechanisms is key for building larger systems and many people still get it wrong. 
The core ideas to understand are:
- **Contiguous tensors** store data in a single, row-major memory layout.
- Many ops (especially slicing like `x[:, ::2]`, `transpose`, `permute`) often create **non-contiguous** tensors (no copy but different strides).
- `view(...)` is **zero-copy** but typically requires **contiguous** memory → may throw an error.
- `reshape(...)` tries to return a view, but if the tensor is non-contiguous it will **allocate/copy**.
- `contiguous()` forces a contiguous copy when the tensor isn’t contiguous.

If you *need* a view after reordering dims: call `x = x.contiguous()` first (this makes a contiguous copy).

In [9]:
def reshape_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """Reshape tensor to new_shape (may return a view or a copy)."""
    x = torch.reshape(x,shape=new_shape)
    return x
    # TODO: implement
    raise NotImplementedError

x = torch.arange(6)
y = reshape_tensor(x, (2, 3))

In [12]:
def view_tensor(x: torch.Tensor, new_shape: Sequence[int]) -> torch.Tensor:
    """View tensor as new_shape (requires contiguous memory and doesn't allocate new memory for the tensor data)."""
    # TODO: implement
    x = x.contiguous()
    x = torch.reshape(x,new_shape)
    return x

y_view = view_tensor(x, (2, 3))

In [13]:
def flatten_from_dim(x: torch.Tensor, start_dim: int = 0) -> torch.Tensor:
    """Flatten a tensor starting from start_dim into a single dimension."""
    x = torch.flatten(x, start_dim=start_dim, end_dim=-1)
    return x
    # TODO: implement
    raise NotImplementedError

x2 = torch.randn(2, 3, 4)
flat = flatten_from_dim(x2, start_dim=1)

In [14]:
def add_singleton_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Insert a size-1 dimension at position dim."""
    x = x.unsqueeze(dim)
    return x
    # TODO: implement
    raise NotImplementedError

x3 = torch.randn(5, 7)
x3s = add_singleton_dim(x3, dim=1)

In [15]:
def remove_singleton_dims(x: torch.Tensor, dim: int | None = None) -> torch.Tensor:
    """Remove size-1 dimensions."""
    x = x.squeeze()
    return x
    # TODO: implement
    raise NotImplementedError

x4 = torch.randn(2, 1, 3)
x4s = remove_singleton_dims(x4)

In [16]:
def transpose_last_two(x: torch.Tensor) -> torch.Tensor:
    """Swap the last two dimensions of x."""
    # TODO: implement
    x = torch.transpose(x, -2,-1)
    return x
    raise NotImplementedError

x6 = torch.randn(2, 3, 4)
x6t = transpose_last_two(x6)

In [17]:
def permute_bhwc_to_bchw(x: torch.Tensor) -> torch.Tensor:
    """Convert (B, H, W, C) tensor into (B, C, H, W)."""
    x = x.permute(0,3,1,2)
    return x
    # TODO: implement
    raise NotImplementedError

x7 = torch.randn(8, 32, 32, 3)
x7p = permute_bhwc_to_bchw(x7)

In [19]:
def make_contiguous(x: torch.Tensor) -> torch.Tensor:
    """Check if tensor is contiguous and if not make contiguous."""
    if x.is_contiguous():
        return x
    else:
        x = x.contiguous
        return x

x8 = torch.randn(4, 6)[:, ::2]
x8c = make_contiguous(x8)

## Indexing
Now that we know how to create tensors and manipulate them we need to understand how we can extract certain components from them using indexing. 
- Basic slicing (`x[a:b]`) returns a view when possible.
- “Fancy” indexing (lists/tensors of indices) usually allocates a new tensor.
- In-place vs out-of-place matters: if a function says “return a copy, leave the input unchanged”, you need `clone()`.

In [21]:
def slice_rows(x: torch.Tensor, start: int, end: int) -> torch.Tensor:
    """Slice rows in a 2D tensor: x[start:end, :]."""
    x = x[start:end, :]
    return x
    raise NotImplementedError

x = torch.arange(12).reshape(4, 3)
rows = slice_rows(x, 1, 3)
print(rows)
print(x)

tensor([[3, 4, 5],
        [6, 7, 8]])
tensor([[ 0,  1,  2],
        [ 3,  4,  5],
        [ 6,  7,  8],
        [ 9, 10, 11]])


In [23]:
def select_columns(x: torch.Tensor, cols: Sequence[int]) -> torch.Tensor:
    """Select specific columns from a 2D tensor."""
    x = x[:, cols]
    return x
    # TODO: implement
    raise NotImplementedError

cols = select_columns(x, [0, 2])
print(cols)

tensor([[ 0,  2],
        [ 3,  5],
        [ 6,  8],
        [ 9, 11]])


In [26]:
def get_diagonal(x: torch.Tensor) -> torch.Tensor:
    """Get the diagonal of a 2D tensor."""
    x = torch.diagonal(x)
    return x

d = get_diagonal(torch.tensor([[1, 2], [3, 4]]))
print(d)

tensor([1, 4])


In [32]:
def set_subtensor(x: torch.Tensor, row_idx: int, col_idx: int, value: float) -> torch.Tensor:
    """Return a copy of x where x[row_idx, col_idx] is set to value."""
    x[row_idx, col_idx] = value
    return x
    # TODO: implement
    raise NotImplementedError

base = torch.zeros(2, 2)
out = set_subtensor(base, 1, 1, 5.0)
print(out)

tensor([[0., 0.],
        [0., 5.]])


In [34]:
def gather_rows(x: torch.Tensor, row_indices: torch.Tensor) -> torch.Tensor:
    x = torch.index_select(x , 0 , row_indices)
    return x
    """Gather (concat) rows from x using row_indices."""
    


x2 = torch.tensor([[10, 11], [20, 21], [30, 31]])
idx = torch.tensor([2, 0])
gathered = gather_rows(x2, idx)
print(x2)
print(idx)
print(gathered)

tensor([[10, 11],
        [20, 21],
        [30, 31]])
tensor([2, 0])
tensor([[30, 31],
        [10, 11]])


## Broadcasting and reducing
Now we're covering a pytorch mechanism that lets you apply elementwise ops without using python loops. It's important to understand how it works to trace your shapes in complicated systems. The broadcasting rules to know are:
- Dimensions align from the **right**.
- A dimension can broadcast if it’s equal or one of them is **1**.

### Reduction ops and `keepdim`

When you reduce over a dimension (e.g. `sum`, `mean`, `max`), PyTorch can either:

- **remove** the reduced dimension (`keepdim=False`, default), or
- **keep** it as size 1 (`keepdim=True`)

Keeping the dimension is often helpful because it makes broadcasting back “just work”.

#### Shape diagram examples

Assume `x` has shape `(B, T, D)`:

**Sum over time**
- `x.sum(dim=1)` → shape `(B, D)`
- `x.sum(dim=1, keepdim=True)` → shape `(B, 1, D)`

**Mean over features**
- `x.mean(dim=2)` → shape `(B, T)`
- `x.mean(dim=2, keepdim=True)` → shape `(B, T, 1)`

#### Why `keepdim=True` helps with broadcasting

Example: center `x` by subtracting the mean over `T`

- If `m = x.mean(dim=1)` has shape `(B, D)`, then `x - m` **fails** (shapes `(B,T,D)` and `(B,D)` don't align).
- If `m = x.mean(dim=1, keepdim=True)` has shape `(B,1,D)`, then `x - m` **works** via broadcasting.

In [37]:
def sum_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Sum tensor values along dimension dim."""
    x = x.sum(dim=dim, keepdim=keepdim)
    return x
    # TODO: implement
    raise NotImplementedError

x = torch.ones(2, 3)
y = sum_over_dim(x, dim=1)
y.shape

torch.Size([2])

In [39]:
def mean_over_dim(x: torch.Tensor, dim: int, keepdim: bool = False) -> torch.Tensor:
    """Mean along dimension dim."""
    x = x.mean(dim=dim, keepdim=keepdim)
    return x
    # TODO: implement
    raise NotImplementedError

x2 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y2 = mean_over_dim(x2, dim=0)
y2.shape

torch.Size([2])

In [42]:
def max_over_dim(x: torch.Tensor, dim: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Max values and argmax indices along dimension dim."""
    result = torch.max(x, dim=dim)
    return result.values, result.indices


x3 = torch.tensor([[1.0, 5.0], [3.0, 2.0]])
values, idx = max_over_dim(x3, dim=1)
print(values)
print(idx)

tensor([5., 3.])
tensor([1, 0])


In [49]:
x = torch.tensor([[1.0, 5.0], [3.0, 2.0]])
def argmax_over_dim(x: torch.Tensor, dim: int) -> torch.Tensor:
    """Argmax indices along dimension dim."""
    x = torch.argmax(x, dim=dim)
    return x

idx2 = argmax_over_dim(x, dim=1)
print(x.shape)
print(idx2.shape)

torch.Size([2, 2])
torch.Size([2])


In [52]:
def broadcast_add_vector(x: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """Add a vector v to each row of a 2D tensor x using broadcasting."""
    # TODO: implement
    bdc = torch.add(v,x)
    return bdc
    raise NotImplementedError


x4 = torch.zeros(3, 2)
v = torch.tensor([10.0, 20.0])
y4 = broadcast_add_vector(x4, v)

y4.shape

torch.Size([3, 2])

## Vectorization
We want to avoid slow (due to per-iteration overhead) python loops as much as possible and pytorch gives us many tools to avoid it. We cover these basics:
- `cat` vs `stack` (concatenate existing dims vs create a new dim)
- `repeat` vs `expand`
- `scatter_add` / `index_add` for accumulation
- `where` for conditional selection

### `expand` vs `repeat`

- `repeat(...)` **copies** data → larger tensor with independent storage.
- `expand(...)` **does not copy** data → it creates a *view* with clever strides.

This has two important implications:

1) `expand` only works when expanding a **size-1 dimension** (broadcasting a singleton).
2) The expanded tensor may have **many positions pointing to the same memory**.  
   Modifying the expanded tensor can therefore produce surprising results (multiple rows change).

Rule of thumb:
- Use `expand` for read-only broadcasting.
- Use `repeat` if you truly need independent copies.


NOTE: We implore you to write your own quick checks from now on for calling the functions and checking their output. As before you are still required to fill in the TODOs in each function.

In [58]:
def concat_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    result = torch.cat(tensors, dim=dim)
    return result

x4 = torch.zeros(3, 2)
v = torch.zeros(3, 2)
y4 = concat_tensors((x4, v))
print(y4)

tensor([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]])


In [62]:
def stack_tensors(tensors: Sequence[torch.Tensor], dim: int = 0) -> torch.Tensor:
    """Stack tensors along a new dimension dim."""
    # TODO: implement
    stacked = torch.stack(tensors, dim=dim)
    return stacked
    raise NotImplementedError

x4 = torch.zeros(3, 2)
v = torch.zeros(3, 2)
y4 = stack_tensors((x4, v))
print(y4)

a = make_randn((2, 3), seed=123, dtype=torch.float32)
print(a)

tensor([[[0., 0.],
         [0., 0.],
         [0., 0.]],

        [[0., 0.],
         [0., 0.],
         [0., 0.]]])
tensor([[-0.1115,  0.1204, -0.3696],
        [-0.2404, -1.1969,  0.2093]])


In [66]:
def repeat_tensor(x: torch.Tensor, repeats: Sequence[int]) -> torch.Tensor:
    """Repeat tensor along each dimension."""
    x = x.repeat(repeats)
    return x
    # TODO: implement
    raise NotImplementedError

b = make_randn((2, 3), seed=123, dtype=torch.float32)
c = repeat_tensor(b, [2,3])
print(c)

tensor([[-0.1115,  0.1204, -0.3696, -0.1115,  0.1204, -0.3696, -0.1115,  0.1204,
         -0.3696],
        [-0.2404, -1.1969,  0.2093, -0.2404, -1.1969,  0.2093, -0.2404, -1.1969,
          0.2093],
        [-0.1115,  0.1204, -0.3696, -0.1115,  0.1204, -0.3696, -0.1115,  0.1204,
         -0.3696],
        [-0.2404, -1.1969,  0.2093, -0.2404, -1.1969,  0.2093, -0.2404, -1.1969,
          0.2093]])


In [84]:
def expand_tensor(x: torch.Tensor, *sizes: int) -> torch.Tensor:
    """Expand tensor to a larger size without copying data.(Sizes can be -1 to keep original dimension.)"""
    x = x.expand(*sizes)
    return x

b = make_randn((1,1), seed=123, dtype=torch.float32)
c = expand_tensor(b, (2,3))
print(c)

import torch

# A single RGB color filter (Red: 255, Green: 0, Blue: 0)
# Shape is (3, 1, 1) -> 3 channels, 1 pixel high, 1 pixel wide
color_filter = torch.tensor([[[255]], [[0]], [[0]]])
print("Original shape:", color_filter.shape)

# You want to apply this color filter to a 4x4 pixel image.
# You MUST expand the 1x1 pixels to 4x4 so the math works.
expanded_filter = color_filter.expand(3, 4, 4)

print("\nExpanded shape:", expanded_filter.shape)
print("\nLook at the Red Channel now (expanded from 1 value to a 4x4 grid):")
print(expanded_filter[0]) # Prints a 4x4 grid of 255s


tensor([[-0.1115, -0.1115, -0.1115],
        [-0.1115, -0.1115, -0.1115]])
Original shape: torch.Size([3, 1, 1])

Expanded shape: torch.Size([3, 4, 4])

Look at the Red Channel now (expanded from 1 value to a 4x4 grid):
tensor([[255, 255, 255, 255],
        [255, 255, 255, 255],
        [255, 255, 255, 255],
        [255, 255, 255, 255]])


In [87]:
def cumsum_over_dim(x: torch.Tensor, dim: int = 0) -> torch.Tensor:
    """Cumulative sum along dim."""
    x = torch.cumsum(x, dim=dim)
    return x

b = make_randn((2, 3), seed=123, dtype=torch.float32)
c = cumsum_over_dim(b, dim=0)
print(b, b.shape)
print(c, c.shape)

tensor([[-0.1115,  0.1204, -0.3696],
        [-0.2404, -1.1969,  0.2093]]) torch.Size([2, 3])
tensor([[-0.1115,  0.1204, -0.3696],
        [-0.3519, -1.0766, -0.1604]]) torch.Size([2, 3])


In [88]:
def where_select(mask: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Elementwise select: return a where mask is True else b. mask must be broadcastable to a and b."""
    x = torch.where(mask,a, b)
    return x

x = torch.tensor([10, 20, 30, 40])
y = torch.tensor([1, 2, 3, 4])
mask = torch.tensor([True, False, True, False])

z = where_select(mask, x, y)
print(z)

tensor([10,  2, 30,  4])


In [106]:
def one_hot(indices: torch.Tensor, num_classes: int, dtype: torch.dtype | None = None) -> torch.Tensor:
    """
    Create one-hot encodings.
    Output is a tensor of the same shape as indices with an added dimension of size num_classes at the end, 
    where the value along that dimension is 1 if it matches the index and 0 otherwise.

    Shapes:
    - indices: (...,) integer tensor
    Return:
    - out: (..., num_classes)

    Requirements:
    - Must work for arbitrary leading shape.
    - No Python loops.
    """
    one_h = torch.nn.functional.one_hot(indices, num_classes=num_classes)
    one_h = one_h.to(dtype)

    return one_h


    
# 1. Pick a small indices tensor by hand
indices = torch.LongTensor([2,3,8])
num_classes = 9


# 3. Call your function and compare
out = one_hot(indices, num_classes, torch.float32)
print(out)


# # 4. Now test the part you haven't actually exercised yet: arbitrary leading shape
# indices_2d = torch.tensor([[___, ___], [___, ___]])
# out_2d = one_hot(indices_2d, num_classes)
# print(out_2d.shape)  # what shape do you expect this to be?

# # 5. Test dtype — does this currently work the way you'd expect?
# out_float = one_hot(indices, num_classes, dtype=torch.float32)
# print(out_float.dtype)

tensor([[0., 0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1.]])


In [107]:
def scatter_add_1d(
    values: torch.Tensor, indices: torch.Tensor, size: int
) -> torch.Tensor:
    """
    Sum `values` into an output vector at positions `indices`.

    Shapes:
    - values: (N,)
    - indices: (N,) integer indices in [0, size)
    Return:
    - out: (size,) with same dtype and device as values

    Requirement:
    - no Python loops
    """
    out = torch.zeros(size, dtype=values.dtype, device=values.device)
    out.index_add_(0, indices, values)
    return out


# 1. Run a test case with your data
result = scatter_add_1d(torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0, 6.0]), torch.tensor([0, 1, 0, 2, 3, 1]), size=4)

# 2. Print the output to verify it matches tensor([4., 8., 4., 5.])
print("Test Output:", result)

Test Output: tensor([4., 8., 4., 5.])


In [ ]:
def batched_token_histogram(tokens: torch.Tensor, vocab_size: int) -> torch.Tensor:
    """
    Count token occurrences per batch item.

    Shapes:
    - tokens: (B, T) int64
    Return:
    - counts: (B, vocab_size) where counts[b, v] = number of times token v appears in tokens[b] 

    Requirements:
    - No Python loops over B or T.
    """
    B,T = tokens.shape

vocab_size = 30522
batch = batched_token_histogram(x,vocab_size)
print(batch)

tensor([1, 1, 1, 1])


In [123]:
import torch


def batched_token_histogram(tokens: torch.Tensor, vocab_size: int) -> torch.Tensor:
    """Count token occurrences per batch item without Python loops.

    Shapes:
        - tokens: (B, T) int64
    Return:
        - counts: (B, vocab_size) where counts[b, v] = number of times token v
          appears in tokens[b]
    """
    B, T = tokens.shape

    # Allocate the final output tensor directly
    counts = torch.zeros(B, vocab_size, dtype=torch.int64, device=tokens.device)

    # Create a matching tensor of ones to add for every token occurrence
    src = torch.ones((B, T), dtype=torch.int64, device=tokens.device)

    # In-place accumulation along dimension 1 (vocab_size) using the token IDs as indices
    counts.scatter_add_(1, tokens, src)

    return counts


# --- Verification Example ---
# Mock data: Batch of 2 rows, Sequence length of 4, Vocabulary size 30522
x = torch.tensor([[101, 5000, 5000, 102], [2000, 101, 2000, 2000]], dtype=torch.int64)

vocab_size = 30522
batch = batched_token_histogram(x, vocab_size)
print(batch)
# # Verify results for specific tokens
# print(f"Batch 0, token 101 count: {batch[0, 101].item()}")  # Expected: 2
# print(f"Batch 1, token 101 count: {batch[1, 101].item()}")  # Expected: 3
# print(f"Output shape: {batch.shape}")  # Expected: torch.Size([2, 30522])
# Find indices (batch_idx, token_id) where counts are greater than 0
indices = (batch > 0).nonzero(as_tuple=True)

for b_idx, token_id in zip(*indices):
    count = batch[b_idx, token_id].item()
    print(f"Batch item {b_idx} | Token ID: {token_id} | Count: {count}")


tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]])
Batch item 0 | Token ID: 101 | Count: 1
Batch item 0 | Token ID: 102 | Count: 1
Batch item 0 | Token ID: 5000 | Count: 2
Batch item 1 | Token ID: 101 | Count: 1
Batch item 1 | Token ID: 2000 | Count: 3


In [126]:
def masked_mean(x: torch.Tensor, mask: torch.Tensor, dim: int) -> torch.Tensor:
    """
    Mean over `dim` considering only mask==True entries.

    Convention:
    - mask: bool tensor broadcastable to x
    - mask==True means "keep this entry"

    Return: same shape as x.mean(dim=dim)

    Requirements:
    - Avoid division by zero: if all mask are False along `dim`, define mean as 0.
    """
    mask_float = mask.float()

    sum_tensor = torch.sum(x * mask_float, dim=dim)
    mask_count = torch.clamp(torch.sum(mask_float, dim=dim), min=1.0)
    result = sum_tensor/mask_count
    return result

x = torch.tensor([[10.0, 20.0, 30.0], [5.0, 5.0, 5.0]])
mask = torch.tensor([[True, True, False], [True, True, True]])
dim = 0

result = masked_mean(x,mask,dim)
print(result)

tensor([ 7.5000, 12.5000,  5.0000])


## Einsum warmup
Now that you’re comfortable with shapes and broadcasting, we’ll introduce `torch.einsum`, a concise way to express tensor operations by explicitly naming axes and summing over repeated indices.

### The idea
You describe each input tensor by labeling its dimensions with letters, e.g.
- `x: (B, T, D)` → `"btd"`
- `W: (D, H)`    → `"dh"`

Then you tell einsum what output labels you want:
- `"btd,dh->bth"`

### Rules of einsum
1) **Same letter = same axis** (must match in size, except broadcastable size-1).
2) **Repeated letters are summed over** (a “contraction”).
3) **Letters that appear in the output are kept** (in that order).
4) You can **reorder axes** just by changing the output label order.

### Tiny cheat sheet
- Sum over an axis: `"btd->bt"` (sums over `d`)
- Transpose: `"ij->ji"`
- Dot product: `"d,d->"` or batched `"btd,btd->bt"`
- Matrix multiply: `"ik,kj->ij"`
- Batched matmul: `"bij,bjk->bik"`
- Outer product: `"i,j->ij"`

### How to derive an einsum (recommended workflow)
1) Write down shapes with named axes (e.g. `q: b h t d`, `k: b h s d`).
2) Decide which axes you want to **sum over** (give them the same letter in both inputs).
3) Decide which axes you want to **keep** in the output (write them after `->`).

In this section, you’ll use einsum to implement building blocks that show up in attention:
- linear projections (`x @ W`)
- dot products
- attention score matrices (`QKᵀ`)
- applying attention weights (`softmax(scores) @ V`)

NOTE: For these exercises you are required to use `torch.einsum` not `matmul` (we check). You are also not required to understand the attention mechanism at this point and the exercises are sovable without. It is good however, to remember the implementations in this exercise for future implementations.

In [129]:
def einsum_linear_btd_dh_to_bth(x: torch.Tensor, W: torch.Tensor) -> torch.Tensor:
    """
    Linear projection using einsum.

    Shapes:
    - x: (B, T, D)
    - W: (D, H)
    Return:
    - y: (B, T, H)
    """
    y = torch.einsum('btd,dh->bth', x,W)
    return y

x = torch.tensor([[[1.0, 2.0]]])   # shape (1,1,2)
w = torch.tensor([[3.0], [4.0]])   # shape (2,1)

y = einsum_linear_btd_dh_to_bth(x,w)
print(y)
y.shape

tensor([[[11.]]])


torch.Size([1, 1, 1])

In [134]:
def einsum_pairwise_dot(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """
    Pairwise dot product between x and y.

    Shapes:
    - x: (B, T, D)
    - y: (B, T, D)
    Return:
    - dots: (B, T) where dots[b,t] = dot(x[b,t], y[b,t])
    """
    dots = torch.einsum('btd,btd -> bt', x,y)
    return dots

x = make_randn((2, 3, 5), seed=42, dtype=torch.bfloat16)
y = make_randn((2, 3, 5), seed=42, dtype=torch.bfloat16)
dots = einsum_pairwise_dot(x,y)
dots.shape
print(dots)

tensor([[11.6250,  7.3750,  5.6562],
        [ 2.2031,  7.1562,  2.6406]], dtype=torch.bfloat16)


In [139]:
def einsum_qk_scores(q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
    """
    Compute attention scores QK^T using einsum.

    Shapes:
    - q: (B, H, T, Dh)
    - k: (B, H, T, Dh)
    Return:
    - scores: (B, H, T, T) where scores[b,h,i,j] = dot(q[b,h,i], k[b,h,j])
    """
    scores = torch.einsum('bhid,bhjd->bhij',q,k)
    return scores

q = make_randn((2, 3, 5,3), seed=42, dtype=torch.bfloat16)
k = make_randn((2, 3, 5,3), seed=42, dtype=torch.bfloat16)
scores = einsum_qk_scores(q,k)
print(scores)
scores.shape

tensor([[[[ 6.7500, -4.1875, -3.1406,  1.3281, -2.9219],
          [-4.1875,  6.4375, -0.0674, -2.0156,  2.1094],
          [-3.1406, -0.0674,  3.1406,  1.6172,  1.5078],
          [ 1.3281, -2.0156,  1.6172,  4.8438,  0.1035],
          [-2.9219,  2.1094,  1.5078,  0.1035,  1.4297]],

         [[ 3.2969,  0.4629,  1.8672,  3.0000,  0.6328],
          [ 0.4629,  1.0156, -1.4609, -0.5312, -0.7969],
          [ 1.8672, -1.4609,  4.6250,  3.4375,  1.3281],
          [ 3.0000, -0.5312,  3.4375,  3.6875,  1.4375],
          [ 0.6328, -0.7969,  1.3281,  1.4375,  1.8438]],

         [[ 2.7188, -2.3438, -0.8672, -2.5156,  1.4688],
          [-2.3438,  3.7656,  0.3438,  2.7344, -1.2891],
          [-0.8672,  0.3438,  0.3750,  0.7188, -0.4160],
          [-2.5156,  2.7344,  0.7188,  4.0000,  0.3457],
          [ 1.4688, -1.2891, -0.4160,  0.3457,  2.7500]]],


        [[[ 6.2812, -1.1719,  1.1406, -0.1338, -0.1758],
          [-1.1719,  1.2734, -0.0688,  1.1172,  0.0618],
          [ 1.1406, -0.

torch.Size([2, 3, 5, 5])

In [ ]:
def einsum_apply_attention(weights: torch.Tensor, v: torch.Tensor) -> torch.Tensor:
    """
    Apply attention weights to values using einsum.

    Shapes:
    - weights: (B, H, T, T)
    - v:       (B, H, T, Dh)
    Return: 
    - out:     (B, H, T, Dh) where out[b,h,i] = sum_j weights[b,h,i,j] * v[b,h,j]
    """
    out = torch.einsum('bhij,bhjd -> bhid', weights,v)
    return out

weights = make_randn((2, 3, 5,5), seed=42, dtype=torch.bfloat16)
v = make_randn((2, 3, 5,3), seed=42, dtype=torch.bfloat16)
out = einsum_apply_attention(weights,v)
print(out)

tensor([[[[-3.4219,  2.8750,  1.6719],
          [-4.6562,  0.0786, -0.0618],
          [ 1.8750,  0.2734,  3.3125],
          [-3.1250,  2.4531, -0.8555],
          [-1.9297, -3.0469, -5.9688]],

         [[ 1.8828,  3.1562,  1.4375],
          [-3.5312, -2.2188, -3.4688],
          [ 0.0532,  3.2031, -1.0703],
          [-5.0938, -3.2969, -2.4062],
          [-0.3418,  1.5156, -0.8086]],

         [[-0.8398, -1.2031,  2.8594],
          [ 0.8750, -0.6406, -2.3125],
          [-4.3125, -3.0156,  2.4844],
          [-2.0781, -0.4023,  3.3906],
          [-1.3594, -1.9688,  4.0625]]],


        [[[ 1.8047,  1.0781,  1.6641],
          [ 0.1187,  1.2578, -0.4414],
          [-0.5234, -1.9375, -1.4766],
          [-2.9688,  2.2500,  0.5859],
          [ 2.9531,  0.2363, -0.9375]],

         [[ 3.2344,  2.9531, -0.4707],
          [ 0.6094,  2.4531, -0.1338],
          [-3.5156, -1.4062, -0.8984],
          [ 1.1484,  2.6562, -0.4043],
          [ 0.2910,  0.1602,  1.2578]],

         [[-2

## Attention Fundamentals
This exercise introduces some building blocks of the attention mechanism which we will encounter extensively throughout the course. It's not yet required for you to fully understand the mechanism to implement the exercises. However, it's good to remember these building blocks for the future. 

To complete the exercises you should familiarize yourself with these topics:
- Stable softmax read: https://jaykmody.com/blog/stable-softmax/
- Masking: typically this means setting masked logits to -inf *before* softmax.
- For attention: causal masks are upper-triangular (no attending to the future).

In [ ]:
def stable_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Numerically stable softmax along `dim`.

    Requirements:
    - Must not overflow for large values in x.
    - Output sums to 1 along `dim`.
    """
    x = x - torch.max(x)
    return torch.softmax(x, dim=dim)


v = make_randn((2, 3, 5,3), seed=42, dtype=torch.bfloat16)
print(stable_softmax(v))



tensor([[[[0.0430, 0.0275, 0.0154],
          [0.0008, 0.0123, 0.0018],
          [0.0060, 0.0013, 0.0029],
          [0.0325, 0.0042, 0.0015],
          [0.0030, 0.0035, 0.0029]],

         [[0.0134, 0.0322, 0.0053],
          [0.0038, 0.0096, 0.0029],
          [0.0183, 0.0138, 0.0332],
          [0.0225, 0.0228, 0.0114],
          [0.0237, 0.0049, 0.0065]],

         [[0.0048, 0.0146, 0.0016],
          [0.0026, 0.0050, 0.0347],
          [0.0085, 0.0041, 0.0084],
          [0.0029, 0.0013, 0.0168],
          [0.0026, 0.0034, 0.0017]]],


        [[[0.0520, 0.0018, 0.0038],
          [0.0025, 0.0032, 0.0067],
          [0.0106, 0.0038, 0.0204],
          [0.0028, 0.0030, 0.0015],
          [0.0064, 0.0058, 0.0121]],

         [[0.0057, 0.0393, 0.0019],
          [0.0248, 0.0264, 0.0146],
          [0.0571, 0.0105, 0.0088],
          [0.0051, 0.0022, 0.0225],
          [0.0052, 0.0105, 0.0063]],

         [[0.0044, 0.0016, 0.0034],
          [0.0106, 0.0105, 0.0194],
          [0.006

In [151]:
def masked_fill_tensor(x: torch.Tensor, mask: torch.Tensor, value: float) -> torch.Tensor:
    """
    Return a copy of x where positions with mask == True are replaced by `value`.
    
    Requirements:
    - mask must be broadcastable to x.
    - do NOT modify x in-place.
    """
    x = x.masked_fill(mask,value=value)
    return x

 
x = torch.tensor([1,2,3])
mask = torch.tensor([True, False,False])   # bool, same shape
value = 0


expected = torch.tensor([0,2,3])   # work this out by hand first

out = masked_fill_tensor(x, mask, value)
print(out)
assert torch.equal(out, expected)

tensor([0, 2, 3])


In [ ]:
def masked_softmax(x: torch.Tensor, mask: torch.Tensor, dim: int = -1) -> torch.Tensor:
    """
    Softmax over x with a boolean mask.

    Convention:
    - mask == True means "invalid and must receive probability 0".
    - Do masking before softmax (i.e., set invalid logits to a large negative).”

    Requirements:
    - Must be numerically stable.
    - Output must be exactly 0 where mask==True.
    - If all entries are masked along `dim`, return all zeros along `dim`.
    - You may reuse functions you implemented above.
    """
    masked_logits = x.masked_fill(mask,-1e9)
    probs = stable_softmax(masked_logits, dim=dim)
    probs = probs.masked_fill(mask, 0.0)
    return probs   

In [2]:
import torch
def make_causal_mask(T: int, device: torch.device | str | None = None) -> torch.Tensor:
    """
    Create a causal (future-masking) boolean mask of shape (T, T).

    Convention:
    - mask[i, j] == True  => position (i attends to j) is NOT allowed (j is in the future)
    - mask[i, j] == False => allowed

    So this is an upper-triangular mask above the diagonal.

    Return:
    - mask: boolean tensor on the specified device

    Example (T=4):
        [[F, T, T, T],
         [F, F, T, T],
         [F, F, F, T],
         [F, F, F, F]]s
    """
    mask = torch.triu(torch.ones(T,T),diagonal=1)
    mask = mask.bool()
    mask = mask.to(device)
    return mask

torch.triu(torch.ones(4,4),diagonal=0)

tensor([[1., 1., 1., 1.],
        [0., 1., 1., 1.],
        [0., 0., 1., 1.],
        [0., 0., 0., 1.]])

In [154]:
def apply_causal_mask(attn_logits: torch.Tensor, value: float = -1e9) -> torch.Tensor:
    """
    Apply a causal mask to attention logits.

    Expected shapes:
    - attn_logits: (..., T, T)

    Returns:
    - masked logits (same shape) where masked positions have been set to `value`.

    Notes:
    - Create a causal mask for the final two dims.
    - Broadcast it across leading dims.
    - You may reuse functions declared above.
    """
    mask = make_causal_mask(attn_logits.shape[-1], attn_logits.device)
    attn_logits = masked_fill_tensor(attn_logits, mask, value)
    return attn_logits